In [1]:
import os
print(os.listdir(r"C:\Master Physische Geographie\Canada\Masterarbeit\results\test_set_evaluation"))

['test_fall', 'test_fall.yaml', 'test_set_evaluation', 'test_set_results_summary.csv', 'test_summer', 'test_summer.yaml']


In [1]:
# wilcoxon_summer_vs_fall.py
#
# Purpose: Wilcoxon signed-rank test to determine whether the summer/fall
# mAP50 performance gap for RT-DETR v7 is statistically significant
#
# Why Wilcoxon and not paired t-test:
#   Only 5 pairs (one per CV fold) — too few for t-test normality assumption
#   Wilcoxon is non-parametric, appropriate for small paired samples
#
# Why paired test:
#   The 5 summer folds and 5 fall folds come from the same spatial CV split
#   Fold 1 summer and fold 1 fall use tiles from the same geographic regions
#   This pairing controls for spatial variation between folds
#
# Null hypothesis (H0): summer and fall mAP50 are equal
# Alternative hypothesis (H1): summer mAP50 > fall mAP50
# Significance level: alpha = 0.05

import numpy as np
from scipy.stats import wilcoxon
import pandas as pd
import os

# ── RT-DETR v7 per-fold mAP50 scores ─────────────────────────────────────────
# extracted from results.csv files (maximum mAP50 achieved per fold)
# summer: trained on summer tiles, validated on summer tiles
# fall:   trained on fall tiles, validated on fall tiles
# same 5-fold knndm spatial CV split used for both seasons

v7_summer_folds = [0.5810, 0.6231, 0.6578, 0.6637, 0.6673]  # folds 1-5
v7_fall_folds   = [0.2314, 0.2930, 0.2272, 0.3037, 0.3501]  # folds 1-5

summer = np.array(v7_summer_folds)
fall   = np.array(v7_fall_folds)
diffs  = summer - fall  # positive = summer better in every fold

print("="*60)
print("WILCOXON SIGNED-RANK TEST — RT-DETR v7")
print("Summer vs Fall mAP50 across 5 CV folds")
print("="*60)

print(f"\nPer-fold scores:")
print(f"{'Fold':>6} {'Summer':>10} {'Fall':>10} {'Diff':>10}")
for i in range(5):
    print(f"{i+1:>6} {summer[i]:>10.4f} {fall[i]:>10.4f} {diffs[i]:>10.4f}")

print(f"\nSummary statistics:")
print(f"  Mean summer mAP50:     {summer.mean():.4f} ± {summer.std():.4f}")
print(f"  Mean fall mAP50:       {fall.mean():.4f} ± {fall.std():.4f}")
print(f"  Mean difference:       {diffs.mean():.4f}")
print(f"  Min difference:        {diffs.min():.4f}")
print(f"  Max difference:        {diffs.max():.4f}")
print(f"  All folds summer>fall: {all(diffs > 0)}")

# ── Wilcoxon test ─────────────────────────────────────────────────────────────
# alternative='greater' — one-tailed test: is summer > fall?
stat, p_value = wilcoxon(summer, fall, alternative='greater')

print(f"\nWilcoxon signed-rank test (H1: summer > fall):")
print(f"  Test statistic: {stat:.4f}")
print(f"  p-value:        {p_value:.6f}")

if p_value < 0.001:
    sig_label = "*** (p<0.001)"
elif p_value < 0.01:
    sig_label = "** (p<0.01)"
elif p_value < 0.05:
    sig_label = "* (p<0.05)"
else:
    sig_label = "not significant"

print(f"  Significance:   {sig_label}")

if p_value < 0.05:
    print(f"\n  CONCLUSION: The summer mAP50 is significantly higher than")
    print(f"  fall mAP50 across all 5 CV folds (p={p_value:.4f}).")
    print(f"  The seasonal performance gap of {diffs.mean():.3f} mAP50 points")
    print(f"  is not due to random variation between folds.")
else:
    print(f"\n  CONCLUSION: The difference is not statistically significant.")

# ── also report effect size (rank-biserial correlation) ───────────────────────
# for Wilcoxon, effect size r = Z / sqrt(N)
# approximated here from the W statistic
n = len(summer)
# W statistic from wilcoxon: max possible is n*(n+1)/2
w_max = n * (n + 1) / 2
r_effect = 1 - (2 * stat / w_max)  # rank-biserial correlation
print(f"\n  Effect size (rank-biserial r): {r_effect:.4f}")
print(f"  Interpretation: {'large (r>0.5)' if abs(r_effect) > 0.5 else 'medium (r>0.3)' if abs(r_effect) > 0.3 else 'small'}")

# ── save results ──────────────────────────────────────────────────────────────
output_dir = r"C:\Master Physische Geographie\Canada\Masterarbeit\results\statistical_analysis"
os.makedirs(output_dir, exist_ok=True)

results_df = pd.DataFrame({
    "fold":          [1, 2, 3, 4, 5],
    "summer_mAP50":  v7_summer_folds,
    "fall_mAP50":    v7_fall_folds,
    "difference":    list(diffs),
})

results_df.to_csv(os.path.join(output_dir, "wilcoxon_results.csv"), index=False)

summary_df = pd.DataFrame([{
    "test":            "Wilcoxon signed-rank (summer > fall)",
    "model":           "RT-DETR v7",
    "n_pairs":         5,
    "mean_summer":     round(summer.mean(), 4),
    "mean_fall":       round(fall.mean(), 4),
    "mean_diff":       round(diffs.mean(), 4),
    "W_statistic":     stat,
    "p_value":         round(p_value, 6),
    "significant":     p_value < 0.05,
    "effect_size_r":   round(r_effect, 4),
}])
summary_df.to_csv(os.path.join(output_dir, "wilcoxon_summary.csv"), index=False)
print(f"\nSaved to: {output_dir}")

WILCOXON SIGNED-RANK TEST — RT-DETR v7
Summer vs Fall mAP50 across 5 CV folds

Per-fold scores:
  Fold     Summer       Fall       Diff
     1     0.5810     0.2314     0.3496
     2     0.6231     0.2930     0.3301
     3     0.6578     0.2272     0.4306
     4     0.6637     0.3037     0.3600
     5     0.6673     0.3501     0.3172

Summary statistics:
  Mean summer mAP50:     0.6386 ± 0.0328
  Mean fall mAP50:       0.2811 ± 0.0465
  Mean difference:       0.3575
  Min difference:        0.3172
  Max difference:        0.4306
  All folds summer>fall: True

Wilcoxon signed-rank test (H1: summer > fall):
  Test statistic: 15.0000
  p-value:        0.031250
  Significance:   * (p<0.05)

  CONCLUSION: The summer mAP50 is significantly higher than
  fall mAP50 across all 5 CV folds (p=0.0312).
  The seasonal performance gap of 0.357 mAP50 points
  is not due to random variation between folds.

  Effect size (rank-biserial r): -1.0000
  Interpretation: large (r>0.5)

Saved to: C:\Master P

In [2]:
import pandas as pd
import os

base = r"C:\Master Physische Geographie\Canada\Masterarbeit\results\summer_rtdetr_2cls_v7"
for fold in range(1, 6):
    csv = os.path.join(base, f"fold{fold}", "results.csv")
    df = pd.read_csv(csv)
    df.columns = [c.strip() for c in df.columns]
    best = df["metrics/mAP50(B)"].max()
    print(f"Fold {fold}: {best:.4f}")

Fold 1: 0.5810
Fold 2: 0.6231
Fold 3: 0.6578
Fold 4: 0.6637
Fold 5: 0.6673


In [3]:
import pandas as pd
import os

# fall v7 fold scores
base_fall = r"C:\Master Physische Geographie\Canada\Masterarbeit\results\fall_rtdetr_2cls_v7"
print("Fall v7 fold scores:")
for fold in range(1, 6):
    csv = os.path.join(base_fall, f"fold{fold}", "results.csv")
    df = pd.read_csv(csv)
    df.columns = [c.strip() for c in df.columns]
    best = df["metrics/mAP50(B)"].max()
    print(f"Fold {fold}: {best:.4f}")

# check prediction folder structure
print("\nTest summer folder contents:")
print(os.listdir(r"C:\Master Physische Geographie\Canada\Masterarbeit\results\test_set_evaluation\test_summer"))

print("\nTest fall folder contents:")
print(os.listdir(r"C:\Master Physische Geographie\Canada\Masterarbeit\results\test_set_evaluation\test_fall"))

Fall v7 fold scores:
Fold 1: 0.2314
Fold 2: 0.2930
Fold 3: 0.2272
Fold 4: 0.3037
Fold 5: 0.3501

Test summer folder contents:
['BoxF1_curve.png', 'BoxPR_curve.png', 'BoxP_curve.png', 'BoxR_curve.png', 'confusion_matrix.png', 'confusion_matrix_normalized.png', 'labels', 'val_batch0_labels.jpg', 'val_batch0_pred.jpg', 'val_batch1_labels.jpg', 'val_batch1_pred.jpg', 'val_batch2_labels.jpg', 'val_batch2_pred.jpg']

Test fall folder contents:
['BoxF1_curve.png', 'BoxPR_curve.png', 'BoxP_curve.png', 'BoxR_curve.png', 'confusion_matrix.png', 'confusion_matrix_normalized.png', 'labels', 'val_batch0_labels.jpg', 'val_batch0_pred.jpg', 'val_batch1_labels.jpg', 'val_batch1_pred.jpg', 'val_batch2_labels.jpg', 'val_batch2_pred.jpg']


In [5]:
import pandas as pd
import os

experiments = {
    "v7":  ("summer_rtdetr_2cls_v7",  "fall_rtdetr_2cls_v7"),
    "v8":  ("summer_rtdetr_2cls_v8",  "fall_rtdetr_2cls_v8"),
    "v9":  ("summer_rtdetr_2cls_v9",  "fall_rtdetr_2cls_v9"),
    "v10": ("summer_rtdetr_2cls_v10", "fall_rtdetr_2cls_v10"),
    "v12": ("summer_rtdetr_2cls_v12", "fall_rtdetr_2cls_v12"),
}

BASE = r"C:\Master Physische Geographie\Canada\Masterarbeit\results"

for v, (s_folder, f_folder) in experiments.items():
    summer_maps = []
    fall_maps   = []
    for fold in range(1, 6):
        for folder, lst in [(s_folder, summer_maps), (f_folder, fall_maps)]:
            csv = os.path.join(BASE, folder, f"fold{fold}", "results.csv")
            if os.path.exists(csv):
                df = pd.read_csv(csv)
                df.columns = [c.strip() for c in df.columns]
                lst.append(df["metrics/mAP50(B)"].max())
    print(f"{v}: summer_mean={sum(summer_maps)/len(summer_maps):.4f} "
          f"fall_mean={sum(fall_maps)/len(fall_maps):.4f}")
    print(f"   summer: {[round(x,4) for x in summer_maps]}")
    print(f"   fall:   {[round(x,4) for x in fall_maps]}")

v7: summer_mean=0.6386 fall_mean=0.2811
   summer: [np.float64(0.581), np.float64(0.6231), np.float64(0.6578), np.float64(0.6637), np.float64(0.6673)]
   fall:   [np.float64(0.2314), np.float64(0.293), np.float64(0.2272), np.float64(0.3037), np.float64(0.3501)]
v8: summer_mean=0.6455 fall_mean=0.2459
   summer: [np.float64(0.5792), np.float64(0.6195), np.float64(0.6348), np.float64(0.7109), np.float64(0.6833)]
   fall:   [np.float64(0.2186), np.float64(0.2175), np.float64(0.2025), np.float64(0.3035), np.float64(0.2873)]
v9: summer_mean=0.6110 fall_mean=0.2750
   summer: [np.float64(0.5451), np.float64(0.6302), np.float64(0.6577)]
   fall:   [np.float64(0.2264), np.float64(0.2871), np.float64(0.2372), np.float64(0.2903), np.float64(0.3339)]


EmptyDataError: No columns to parse from file

In [6]:
import pandas as pd
import os

experiments = {
    "v7":  ("summer_rtdetr_2cls_v7",  "fall_rtdetr_2cls_v7"),
    "v8":  ("summer_rtdetr_2cls_v8",  "fall_rtdetr_2cls_v8"),
    "v9":  ("summer_rtdetr_2cls_v9",  "fall_rtdetr_2cls_v9"),
    "v10": ("summer_rtdetr_2cls_v10", "fall_rtdetr_2cls_v10"),
    "v12": ("summer_rtdetr_2cls_v12", "fall_rtdetr_2cls_v12"),
}

BASE = r"C:\Master Physische Geographie\Canada\Masterarbeit\results"

for v, (s_folder, f_folder) in experiments.items():
    summer_maps = []
    fall_maps   = []
    for fold in range(1, 6):
        for folder, lst in [(s_folder, summer_maps), (f_folder, fall_maps)]:
            csv = os.path.join(BASE, folder, f"fold{fold}", "results.csv")
            if not os.path.exists(csv):
                continue
            try:
                df = pd.read_csv(csv)
                if df.empty or len(df.columns) == 0:
                    continue
                df.columns = [c.strip() for c in df.columns]
                if "metrics/mAP50(B)" in df.columns:
                    lst.append(df["metrics/mAP50(B)"].max())
            except Exception as e:
                print(f"  Skipping {folder} fold{fold}: {e}")

    if summer_maps and fall_maps:
        print(f"{v}: summer_mean={sum(summer_maps)/len(summer_maps):.4f} "
              f"fall_mean={sum(fall_maps)/len(fall_maps):.4f}")
        print(f"   summer ({len(summer_maps)} folds): {[round(x,4) for x in summer_maps]}")
        print(f"   fall   ({len(fall_maps)} folds): {[round(x,4) for x in fall_maps]}")
    else:
        print(f"{v}: insufficient data — summer={summer_maps}, fall={fall_maps}")

v7: summer_mean=0.6386 fall_mean=0.2811
   summer (5 folds): [np.float64(0.581), np.float64(0.6231), np.float64(0.6578), np.float64(0.6637), np.float64(0.6673)]
   fall   (5 folds): [np.float64(0.2314), np.float64(0.293), np.float64(0.2272), np.float64(0.3037), np.float64(0.3501)]
v8: summer_mean=0.6455 fall_mean=0.2459
   summer (5 folds): [np.float64(0.5792), np.float64(0.6195), np.float64(0.6348), np.float64(0.7109), np.float64(0.6833)]
   fall   (5 folds): [np.float64(0.2186), np.float64(0.2175), np.float64(0.2025), np.float64(0.3035), np.float64(0.2873)]
v9: summer_mean=0.6110 fall_mean=0.2750
   summer (3 folds): [np.float64(0.5451), np.float64(0.6302), np.float64(0.6577)]
   fall   (5 folds): [np.float64(0.2264), np.float64(0.2871), np.float64(0.2372), np.float64(0.2903), np.float64(0.3339)]
  Skipping fall_rtdetr_2cls_v10 fold4: No columns to parse from file
v10: summer_mean=0.6858 fall_mean=0.2172
   summer (1 folds): [np.float64(0.6858)]
   fall   (3 folds): [np.float64(0.223